# 02 — Traditional Machine Learning

Notebook ini menjalankan eksperimen model traditional ML untuk klasifikasi risiko kredit.
Target yang diprediksi adalah `loan_status`, yaitu `0 = good` dan `1 = default`.

Model yang diuji:
1. Logistic Regression
2. SVM kernel RBF
3. Decision Tree
4. Random Forest `n_estimators=200`
5. XGBoost dengan `gpu_hist` jika GPU tersedia, fallback ke `hist`
6. LightGBM
7. CatBoost dengan categorical feature langsung

Pemilihan model terbaik dilakukan berdasarkan kombinasi `F1-score` dan `AUC-ROC`, bukan accuracy saja, karena data asli imbalanced.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "credit_risk_dataset.csv"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from preprocessing import clean_outliers, handle_missing_values, run_full_pipeline


## 1. Load data hasil preprocessing

Notebook ini memakai output dari `src/preprocessing.py`.
Kalau file processed belum ada, pipeline akan dijalankan otomatis dari dataset raw.

In [ ]:
processed_files = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]
missing_processed = [name for name in processed_files if not (DATA_PROCESSED / name).exists()]

if missing_processed:
    print("Processed files belum lengkap, menjalankan run_full_pipeline()...")
    X_train, X_test, y_train, y_test = run_full_pipeline(str(DATA_RAW))
else:
    X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
    X_test = pd.read_csv(DATA_PROCESSED / "X_test.csv")
    y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze("columns")
    y_test = pd.read_csv(DATA_PROCESSED / "y_test.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train distribution:", y_train.value_counts().to_dict())
print("y_test distribution :", y_test.value_counts().to_dict())
X_train.head()


## 2. Siapkan data khusus CatBoost

CatBoost diminta menangani categorical feature secara langsung. Karena data processed sudah one-hot encoded, CatBoost dibuat dari data raw yang sudah dibersihkan dan diimputasi, lalu training set diseimbangkan dengan `SMOTENC` agar categorical feature tetap categorical.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC

raw_df = pd.read_csv(DATA_RAW)
raw_df = clean_outliers(raw_df)
raw_df = handle_missing_values(raw_df)

cat_features = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file",
]

X_cat = raw_df.drop(columns=["loan_status"])
y_cat = raw_df["loan_status"]

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.20,
    random_state=42,
    stratify=y_cat,
)

cat_indices = [X_cat.columns.get_loc(col) for col in cat_features]
smotenc = SMOTENC(categorical_features=cat_indices, random_state=42)
X_train_cat_bal, y_train_cat_bal = smotenc.fit_resample(X_train_cat, y_train_cat)
X_train_cat_bal = pd.DataFrame(X_train_cat_bal, columns=X_cat.columns)
y_train_cat_bal = pd.Series(y_train_cat_bal, name="loan_status").astype(int)

for col in X_train_cat_bal.columns:
    if col in cat_features:
        X_train_cat_bal[col] = X_train_cat_bal[col].astype(str)
        X_test_cat[col] = X_test_cat[col].astype(str)
    else:
        X_train_cat_bal[col] = pd.to_numeric(X_train_cat_bal[col], errors="coerce")
        X_test_cat[col] = pd.to_numeric(X_test_cat[col], errors="coerce")

print("CatBoost X_train:", X_train_cat_bal.shape)
print("CatBoost y_train:", y_train_cat_bal.value_counts().to_dict())
X_train_cat_bal.head()


## 3. Fungsi evaluasi model

Metrik yang dipakai:
- Accuracy
- Precision
- Recall
- F1-score
- AUC-ROC
- Confusion matrix
- Train time

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)


def get_scores(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)
        y_score = y_score[:, 1] if np.ndim(y_score) == 2 else y_score
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_eval)
    else:
        y_score = y_pred

    return {
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1": f1_score(y_eval, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_eval, y_score),
        "Confusion Matrix": confusion_matrix(y_eval, y_pred),
        "y_pred": y_pred,
        "y_score": y_score,
    }


def plot_confusion_matrix(y_true, y_pred, title):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=["Good", "Default"],
        cmap="Blues",
        values_format="d",
    )
    plt.title(title)
    plt.show()


## 4. Definisi model

XGBoost akan mencoba `gpu_hist` dulu. Jika GPU tidak tersedia, otomatis memakai `hist` supaya tetap bisa jalan di laptop biasa.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


def build_xgb_classifier():
    common_params = dict(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    try:
        probe = XGBClassifier(**common_params, tree_method="gpu_hist")
        probe.fit(X_train.iloc[:256], y_train.iloc[:256])
        print("XGBoost memakai tree_method='gpu_hist'")
        return XGBClassifier(**common_params, tree_method="gpu_hist")
    except Exception as exc:
        print("GPU untuk XGBoost tidak tersedia, fallback ke tree_method='hist'")
        print("Alasan fallback:", str(exc).split("
")[0])
        return XGBClassifier(**common_params, tree_method="hist")


models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="lbfgs", random_state=42),
    "SVM RBF": SVC(kernel="rbf", probability=True, cache_size=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": build_xgb_classifier(),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
    ),
}


## 5. Training dan evaluasi seluruh model

In [ ]:
results = []
fitted_models = {}
model_eval_data = {}

for name, model in models.items():
    print("=" * 80)
    print(f"Training: {name}")
    start_time = time.time()

    if name == "CatBoost":
        model.fit(X_train_cat_bal, y_train_cat_bal, cat_features=cat_features, verbose=False)
        train_time = time.time() - start_time
        scores = get_scores(model, X_test_cat, y_test_cat)
        eval_X = X_test_cat
        eval_y = y_test_cat
        feature_names = list(X_test_cat.columns)
        input_type = "catboost_raw"
    else:
        model.fit(X_train, y_train)
        train_time = time.time() - start_time
        scores = get_scores(model, X_test, y_test)
        eval_X = X_test
        eval_y = y_test
        feature_names = list(X_test.columns)
        input_type = "processed_encoded"

    fitted_models[name] = model
    model_eval_data[name] = {
        "X": eval_X,
        "y": eval_y,
        "feature_names": feature_names,
        "input_type": input_type,
    }

    row = {
        "Model": name,
        "Accuracy": scores["Accuracy"],
        "Precision": scores["Precision"],
        "Recall": scores["Recall"],
        "F1": scores["F1"],
        "AUC-ROC": scores["AUC-ROC"],
        "Train Time": train_time,
    }
    results.append(row)

    print(pd.Series(row).to_string())
    print("
Classification report:")
    print(classification_report(eval_y, scores["y_pred"], target_names=["Good", "Default"], zero_division=0))
    plot_confusion_matrix(eval_y, scores["y_pred"], f"Confusion Matrix — {name}")

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by=["F1", "AUC-ROC"], ascending=False).reset_index(drop=True)
comparison_df


## 6. Comparison table dan pemilihan model terbaik

Model terbaik dipilih dari urutan tertinggi berdasarkan `F1`, lalu `AUC-ROC`.

In [ ]:
comparison_df.to_csv(MODELS_DIR / "traditional_ml_results.csv", index=False)

best_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_eval_info = model_eval_data[best_name]

print("Best model:", best_name)
print(comparison_df.iloc[0].to_string())
comparison_df


## 7. Hyperparameter tuning model terbaik

Tuning menggunakan `RandomizedSearchCV` dengan `n_iter=20` dan `cv=5`.
Untuk CatBoost, tuning memakai `randomized_search` bawaan CatBoost agar categorical feature tetap ditangani langsung.

In [ ]:
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV


def get_param_distribution(model_name):
    distributions = {
        "Logistic Regression": {
            "C": loguniform(1e-3, 100),
            "class_weight": [None, "balanced"],
        },
        "SVM RBF": {
            "C": loguniform(1e-2, 100),
            "gamma": loguniform(1e-4, 1),
            "class_weight": [None, "balanced"],
        },
        "Decision Tree": {
            "max_depth": randint(3, 30),
            "min_samples_split": randint(2, 30),
            "min_samples_leaf": randint(1, 20),
            "criterion": ["gini", "entropy", "log_loss"],
        },
        "Random Forest": {
            "n_estimators": randint(200, 600),
            "max_depth": [None, 6, 10, 14, 18, 24],
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
        },
        "XGBoost": {
            "n_estimators": randint(200, 700),
            "max_depth": randint(3, 10),
            "learning_rate": loguniform(0.01, 0.2),
            "subsample": uniform(0.65, 0.35),
            "colsample_bytree": uniform(0.65, 0.35),
            "min_child_weight": randint(1, 8),
            "reg_lambda": loguniform(0.1, 10),
        },
        "LightGBM": {
            "n_estimators": randint(200, 800),
            "num_leaves": randint(16, 96),
            "learning_rate": loguniform(0.01, 0.2),
            "subsample": uniform(0.65, 0.35),
            "colsample_bytree": uniform(0.65, 0.35),
            "min_child_samples": randint(10, 80),
            "reg_lambda": loguniform(0.1, 10),
        },
    }
    return distributions[model_name]


tuning_start = time.time()

if best_name == "CatBoost":
    tuned_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
    )
    catboost_params = {
        "iterations": [200, 300, 500, 700],
        "depth": [4, 5, 6, 7, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7, 9],
    }
    tuned_model.randomized_search(
        catboost_params,
        X_train_cat_bal,
        y_train_cat_bal,
        cv=5,
        n_iter=20,
        search_by_train_test_split=False,
        refit=True,
        verbose=False,
        plot=False,
    )
    tuned_eval_X = X_test_cat
    tuned_eval_y = y_test_cat
    tuned_feature_names = list(X_test_cat.columns)
    tuned_input_type = "catboost_raw"
else:
    search = RandomizedSearchCV(
        estimator=best_model,
        param_distributions=get_param_distribution(best_name),
        n_iter=20,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )
    search.fit(X_train, y_train)
    tuned_model = search.best_estimator_
    print("Best params:", search.best_params_)
    print("Best CV F1:", search.best_score_)
    tuned_eval_X = X_test
    tuned_eval_y = y_test
    tuned_feature_names = list(X_test.columns)
    tuned_input_type = "processed_encoded"

tuning_time = time.time() - tuning_start
tuned_scores = get_scores(tuned_model, tuned_eval_X, tuned_eval_y)

print("Tuning time:", tuning_time)
print("Tuned model test metrics:")
for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]:
    print(f"{metric}: {tuned_scores[metric]:.4f}")

plot_confusion_matrix(tuned_eval_y, tuned_scores["y_pred"], f"Confusion Matrix — Tuned {best_name}")


## 8. Feature importance top 10

Jika model punya `feature_importances_`, nilai tersebut dipakai. Jika model linear, dipakai absolute coefficient. Jika tidak tersedia, digunakan permutation importance.

In [ ]:
from sklearn.inspection import permutation_importance


def get_importance_values(model, X_eval, y_eval, feature_names):
    if hasattr(model, "feature_importances_"):
        values = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        values = np.abs(np.asarray(model.coef_)).ravel()
    else:
        perm = permutation_importance(
            model,
            X_eval,
            y_eval,
            n_repeats=5,
            random_state=42,
            scoring="f1",
            n_jobs=-1,
        )
        values = perm.importances_mean

    return pd.DataFrame({"Feature": feature_names, "Importance": values})


importance_df = get_importance_values(tuned_model, tuned_eval_X, tuned_eval_y, tuned_feature_names)
importance_df = importance_df.sort_values("Importance", ascending=False).reset_index(drop=True)
top10_importance = importance_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top10_importance["Feature"][::-1], top10_importance["Importance"][::-1])
plt.title(f"Top 10 Feature Importance — Tuned {best_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

top10_importance


## 9. Simpan model terbaik

Model hasil tuning disimpan ke `models/best_model.pkl`.
Metadata dan hasil evaluasi juga disimpan agar bisa dipakai lagi di notebook TabNet atau agent.

In [ ]:
joblib.dump(tuned_model, MODELS_DIR / "best_model.pkl")
joblib.dump(
    {
        "best_model_name": best_name,
        "input_type": tuned_input_type,
        "feature_names": tuned_feature_names,
        "metrics": {metric: tuned_scores[metric] for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]},
        "tuning_time_seconds": tuning_time,
    },
    MODELS_DIR / "best_model_metadata.pkl",
)

print("Saved:")
print(MODELS_DIR / "best_model.pkl")
print(MODELS_DIR / "best_model_metadata.pkl")
print(MODELS_DIR / "traditional_ml_results.csv")


## 10. SHAP explainability untuk 5 applicant contoh

Contoh diambil dari test set dengan campuran class `approve/good` dan `decline/default`.

In [ ]:
from explainability import generate_shap_explanation, get_decline_reasons

# Ambil 3 data default dan 2 data good agar contoh explainability campuran.
y_eval_reset = pd.Series(tuned_eval_y).reset_index(drop=True)
default_indices = list(y_eval_reset[y_eval_reset == 1].index[:3])
good_indices = list(y_eval_reset[y_eval_reset == 0].index[:2])
example_indices = default_indices + good_indices

for idx in example_indices:
    applicant = tuned_eval_X.reset_index(drop=True).iloc[[idx]]
    actual = int(y_eval_reset.iloc[idx])
    pred = int(tuned_model.predict(applicant)[0])

    explanation = generate_shap_explanation(
        tuned_model,
        applicant,
        tuned_feature_names,
        sample_index=0,
    )
    reasons = get_decline_reasons(
        applicant.iloc[0].to_dict(),
        explanation["shap_values"][0],
        threshold=0.03,
    )

    print("=" * 80)
    print(f"Applicant index: {idx}")
    print(f"Actual loan_status: {actual} | Predicted: {pred}")
    print(explanation["summary_text"])
    print("Decline reasons:")
    for reason in reasons:
        print("-", reason)
